# Stress Prediction — 오늘 찾은 모든 개선 통합 (최종)

오늘 검증된 3가지 개선을 전부 합칩니다:
1. gender를 tree 입력에서 제거 (-0.00025)
2. mean_working/diastolic_blood_pressure/cholesterol/glucose winsorize (-0.00064 누적)
3. 근접중복(distance<0.2) 샘플은 최근접이웃 타겟값으로 오버라이드 (-0.00015 추가)

이 세 가지를 다 합치면 이론상 v34 원본(0.148033) 대비 -0.001 근처까지 갈 수 있습니다.
**미리 말씀드립니다: 이걸 다 더해도 0.139에는 크게 못 미칩니다.** 이건 오늘 데이터로
만들 수 있는 최선의 조합을 정확히 확인하기 위한 마지막 테스트입니다.


In [1]:

from __future__ import annotations

import os
from itertools import combinations
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold
from scipy.spatial import cKDTree

TARGET = "stress_score"
ID_COLUMN = "ID"
RANDOM_SEED = 42
N_ESTIMATORS = 1200
TREE_QUANTILE = 0.54
PAIR_QUANTILE = 0.48
PAIR_WEIGHT = 0.28
OVERRIDE_THRESHOLD = 0.2
WINSORIZE_COLUMNS = ["mean_working", "diastolic_blood_pressure", "cholesterol", "glucose"]

DATA_DIR = Path("/Users/bitsaem/Desktop/stress_project/open")
OUTPUT_DIR = Path("/Users/bitsaem/Desktop/stress_project/8_6")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_COPY_COUNTS = {
    "mean_working": 1, "bmi": 4, "cholesterol": 2, "height": 2,
    "glucose": 3, "weight": 2, "cholesterol_glucose_ratio": 2, "bone_density": 4,
}

PAIR_COLUMNS = [
    "mean_working", "bmi", "cholesterol", "height",
    "glucose", "weight", "cholesterol_glucose_ratio", "bone_density",
]


def compute_iqr_bounds(train_frame, columns):
    bounds = {}
    for col in columns:
        q1, q3 = train_frame[col].quantile([0.25, 0.75])
        iqr = q3 - q1
        bounds[col] = (q1 - 1.5 * iqr, q3 + 1.5 * iqr)
    return bounds


def apply_winsorize(frame, bounds):
    result = frame.copy()
    for col, (lower, upper) in bounds.items():
        result[col] = result[col].clip(lower, upper)
    return result


def add_features(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    height_m = result["height"] / 100.0
    sbp = result["systolic_blood_pressure"]
    dbp = result["diastolic_blood_pressure"]
    result["bmi"] = result["weight"] / height_m.pow(2)
    result["pulse_pressure"] = sbp - dbp
    result["mean_arterial_pressure"] = (sbp + 2.0 * dbp) / 3.0
    result["blood_pressure_ratio"] = sbp / (dbp + 1e-6)
    result["cholesterol_glucose_ratio"] = result["cholesterol"] / (result["glucose"] + 1e-6)
    result["missing_count"] = result.isna().sum(axis=1)
    return result


def add_feature_copies(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    for feature, n_copies in FEATURE_COPY_COUNTS.items():
        for i in range(1, n_copies + 1):
            result[f"{feature}__copy{i}"] = result[feature]
    return result


def make_preprocessor(frame: pd.DataFrame) -> ColumnTransformer:
    categorical = frame.select_dtypes(include=["object", "category", "string"]).columns.tolist()
    numerical = frame.select_dtypes(include=["number"]).columns.tolist()
    return ColumnTransformer([
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), categorical),
        ("numerical", SimpleImputer(strategy="median", add_indicator=True), numerical),
    ])


def tree_prediction(train_features, val_features, target) -> np.ndarray:
    train_no_gender = train_features.drop(columns=["gender"])
    val_no_gender = val_features.drop(columns=["gender"])
    weighted_train = add_feature_copies(train_no_gender)
    weighted_val = add_feature_copies(val_no_gender)
    preprocessor = make_preprocessor(weighted_train)
    train_matrix = preprocessor.fit_transform(weighted_train)
    val_matrix = preprocessor.transform(weighted_val)
    model = ExtraTreesRegressor(
        n_estimators=N_ESTIMATORS, min_samples_leaf=1, max_features=1,
        criterion="squared_error", random_state=RANDOM_SEED, n_jobs=-1,
    )
    model.fit(train_matrix, target)
    tree_predictions = np.column_stack([t.predict(val_matrix) for t in model.estimators_])
    return np.quantile(tree_predictions, TREE_QUANTILE, axis=1)


def empirical_rank_transform(train_values, test_values):
    median = np.nanmedian(train_values)
    train_values = np.nan_to_num(train_values, nan=median)
    test_values = np.nan_to_num(test_values, nan=median)
    sorted_train = np.sort(train_values)
    train_rank = np.searchsorted(sorted_train, train_values, side="right") / len(train_values)
    test_rank = np.searchsorted(sorted_train, test_values, side="right") / len(train_values)
    return train_rank.astype(np.float32), test_rank.astype(np.float32)


def pair_prediction(train_features, val_features, target) -> np.ndarray:
    train_rank, val_rank = {}, {}
    for column in PAIR_COLUMNS:
        train_rank[column], val_rank[column] = empirical_rank_transform(
            train_features[column].to_numpy(dtype=float),
            val_features[column].to_numpy(dtype=float),
        )
    target_values = target.to_numpy(dtype=float)
    neighbor_targets = []
    for first, second in combinations(PAIR_COLUMNS, 2):
        distance = (
            np.abs(val_rank[first][:, None] - train_rank[first][None, :])
            + np.abs(val_rank[second][:, None] - train_rank[second][None, :])
        )
        nearest_idx = np.argmin(distance, axis=1)
        neighbor_targets.append(target_values[nearest_idx])
    return np.quantile(np.column_stack(neighbor_targets), PAIR_QUANTILE, axis=1)


def nearest_neighbor_lookup(train_raw, val_raw, target, numeric_cols):
    train_numeric = train_raw[numeric_cols].fillna(train_raw[numeric_cols].median())
    val_numeric = val_raw[numeric_cols].fillna(train_raw[numeric_cols].median())
    scaler = StandardScaler()
    train_scaled = scaler.fit_transform(train_numeric)
    val_scaled = scaler.transform(val_numeric)
    tree = cKDTree(train_scaled)
    nearest_dist, nearest_idx = tree.query(val_scaled, k=1)
    nearest_target = target.to_numpy(dtype=float)[nearest_idx]
    return nearest_dist, nearest_target


def combined_prediction(train_raw_fold, val_raw_fold, target, numeric_cols):
    """gender제거 + winsorize + 근접중복 오버라이드 전부 적용."""
    bounds = compute_iqr_bounds(train_raw_fold, WINSORIZE_COLUMNS)
    train_winsorized = apply_winsorize(train_raw_fold, bounds)
    val_winsorized = apply_winsorize(val_raw_fold, bounds)

    train_feat = add_features(train_winsorized)
    val_feat = add_features(val_winsorized)

    tree_pred = tree_prediction(train_feat, val_feat, target)
    pair_pred = pair_prediction(train_feat, val_feat, target)
    base_pred = np.clip(np.round((1 - PAIR_WEIGHT) * tree_pred + PAIR_WEIGHT * pair_pred, 2), 0.0, 1.0)

    nearest_dist, nearest_target = nearest_neighbor_lookup(train_raw_fold, val_raw_fold, target, numeric_cols)
    override_mask = nearest_dist < OVERRIDE_THRESHOLD
    result = base_pred.copy()
    result[override_mask] = np.clip(np.round(nearest_target[override_mask], 2), 0.0, 1.0)
    return result


## 5-Fold CV — 통합 모델 vs v34 원본

In [2]:

train_raw = pd.read_csv(DATA_DIR / "train.csv")
y = train_raw[TARGET].reset_index(drop=True)
raw_input = train_raw.drop(columns=[TARGET, ID_COLUMN]).reset_index(drop=True)
numeric_cols = raw_input.select_dtypes(include=["number"]).columns.tolist()

cv = KFold(5, shuffle=True, random_state=RANDOM_SEED)
oof_combined = np.zeros(len(y))

for fold, (train_idx, val_idx) in enumerate(cv.split(raw_input), start=1):
    train_raw_fold = raw_input.iloc[train_idx].reset_index(drop=True)
    val_raw_fold = raw_input.iloc[val_idx].reset_index(drop=True)
    y_train = y.iloc[train_idx].reset_index(drop=True)
    oof_combined[val_idx] = combined_prediction(train_raw_fold, val_raw_fold, y_train, numeric_cols)
    print(f"Fold {fold} done")

combined_mae = mean_absolute_error(y, oof_combined)
print()
print(f"### 통합 모델(gender제거+winsorize+오버라이드) CV MAE: {combined_mae:.6f}")
print("(참고) v34 원본 CV MAE: 0.148033")
print(f"(참고) 개선폭: {0.148033 - combined_mae:+.6f}")


Fold 1 done
Fold 2 done
Fold 3 done
Fold 4 done
Fold 5 done

### 통합 모델(gender제거+winsorize+오버라이드) CV MAE: 0.147363
(참고) v34 원본 CV MAE: 0.148033
(참고) 개선폭: +0.000670


## 최종 제출 파일 생성

In [3]:

test_raw = pd.read_csv(DATA_DIR / "test.csv")
submission = pd.read_csv(DATA_DIR / "sample_submission.csv")
test_input = test_raw.drop(columns=[ID_COLUMN]).reset_index(drop=True)

final_prediction = combined_prediction(raw_input, test_input, y, numeric_cols)

output = submission.copy()
output[TARGET] = final_prediction
submit_path = OUTPUT_DIR / "submit_combined_final_0806.csv"
output.to_csv(submit_path, index=False, encoding="utf-8")

print("Saved:", submit_path)
print("Prediction range:", final_prediction.min(), final_prediction.max())
display(output.head(10))


Saved: /Users/bitsaem/Desktop/stress_project/8_6/submit_combined_final_0806.csv
Prediction range: 0.0 1.0


,ID,stress_score
0,TEST_0000,0.61
1,TEST_0001,0.97
2,TEST_0002,0.19
3,TEST_0003,0.53
4,TEST_0004,0.53
5,TEST_0005,0.49
6,TEST_0006,0.78
7,TEST_0007,0.47
8,TEST_0008,0.52
9,TEST_0009,0.88
